In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Student_data/Student_Performance.csv")   # replace with the actual file name

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())

(25000, 16)
   student_id  age  gender school_type parent_education  study_hours  \
0           1   14    male      public    post graduate          3.1   
1           2   18  female      public         graduate          3.7   
2           3   17  female     private    post graduate          7.9   
3           4   16   other      public      high school          1.1   
4           5   16  female      public      high school          1.3   

   attendance_percentage internet_access travel_time extra_activities  \
0                   84.3             yes     <15 min              yes   
1                   87.8             yes     >60 min               no   
2                   65.5              no     <15 min               no   
3                   58.1              no   15-30 min               no   
4                   61.0             yes   30-60 min              yes   

  study_method  math_score  science_score  english_score  overall_score  \
0        notes        42.7           55.4

In [4]:
# 1. Confirm the duplicates are real
print(df['student_id'].nunique())          # how many unique students
print(df[df.duplicated(keep=False)].sort_values('student_id').head(10))

# 2. Remove them
df_clean = df.drop_duplicates().reset_index(drop=True)
print(df_clean.shape)                       # should be about (15000, 16)

# 3. Check ranges for impossible values
print(df_clean.describe())                  # scores should be 0-100, attendance 0-100, age sensible

# 4. Check the text columns for typos or inconsistent labels
for col in ['gender','school_type','parent_education','internet_access',
            'travel_time','extra_activities','study_method','final_grade']:
    print(col, df_clean[col].unique())

# 5. Save the cleaned file
df_clean.to_csv("/content/drive/MyDrive/Student_data/students_clean.csv", index=False)

15000
       student_id  age  gender school_type parent_education  study_hours  \
17372           2   18  female      public         graduate          3.7   
17524           2   18  female      public         graduate          3.7   
23867           2   18  female      public         graduate          3.7   
1               2   18  female      public         graduate          3.7   
2               3   17  female     private    post graduate          7.9   
21076           3   17  female     private    post graduate          7.9   
4               5   16  female      public      high school          1.3   
15557           5   16  female      public      high school          1.3   
24422           5   16  female      public      high school          1.3   
16264           6   19    male      public        no formal          3.8   

       attendance_percentage internet_access travel_time extra_activities  \
17372                   87.8             yes     >60 min               no   
175

In [5]:
import sqlite3
import pandas as pd

# Make sure df_clean exists (if the session restarted, reload the cleaned file)
# df_clean = pd.read_csv("/content/drive/MyDrive/Projects/students_clean.csv")

conn = sqlite3.connect("students.db")
df_clean.to_sql("students", conn, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql_query(sql, conn)

# Check the table loaded
print(q("SELECT COUNT(*) AS total_students FROM students"))

   total_students
0           15000


In [6]:
# 1. Average overall score by study method
print(q("""
SELECT study_method,
       ROUND(AVG(overall_score), 1) AS avg_score,
       COUNT(*) AS students
FROM students
GROUP BY study_method
ORDER BY avg_score DESC
"""))

# 2. Does internet access affect scores?
print(q("""
SELECT internet_access,
       ROUND(AVG(overall_score), 1) AS avg_score,
       ROUND(AVG(attendance_percentage), 1) AS avg_attendance
FROM students
GROUP BY internet_access
"""))

# 3. Grade distribution by school type
print(q("""
SELECT school_type, final_grade, COUNT(*) AS students
FROM students
GROUP BY school_type, final_grade
ORDER BY school_type, final_grade
"""))

# 4. Attendance bands vs average score
print(q("""
SELECT CASE
         WHEN attendance_percentage < 60 THEN '50-59'
         WHEN attendance_percentage < 70 THEN '60-69'
         WHEN attendance_percentage < 80 THEN '70-79'
         WHEN attendance_percentage < 90 THEN '80-89'
         ELSE '90-100'
       END AS attendance_band,
       ROUND(AVG(overall_score), 1) AS avg_score,
       COUNT(*) AS students
FROM students
GROUP BY attendance_band
ORDER BY attendance_band
"""))

    study_method  avg_score  students
0  online videos       64.7      2468
1       coaching       64.5      2422
2       textbook       64.2      2546
3          notes       63.9      2515
4          mixed       63.7      2602
5    group study       63.3      2447
  internet_access  avg_score  avg_attendance
0              no       63.9            75.0
1             yes       64.0            75.0
   school_type final_grade  students
0      private           a       364
1      private           b       818
2      private           c      1852
3      private           d      1950
4      private           e      1712
5      private           f       891
6       public           a       357
7       public           b       820
8       public           c      1845
9       public           d      1820
10      public           e      1666
11      public           f       905
  attendance_band  avg_score  students
0           50-59       56.4      2986
1           60-69       59.8      2986
2

In [7]:
# 5. Study hours bands vs score
print(q("""
SELECT CASE
         WHEN study_hours < 2 THEN '0.5-1.9'
         WHEN study_hours < 4 THEN '2-3.9'
         WHEN study_hours < 6 THEN '4-5.9'
         ELSE '6-8'
       END AS study_band,
       ROUND(AVG(overall_score), 1) AS avg_score,
       COUNT(*) AS students
FROM students
GROUP BY study_band
ORDER BY study_band
"""))

# 6. Parent education vs score
print(q("""
SELECT parent_education,
       ROUND(AVG(overall_score), 1) AS avg_score,
       COUNT(*) AS students
FROM students
GROUP BY parent_education
ORDER BY avg_score DESC
"""))

# 7. Subject comparison
print(q("""
SELECT ROUND(AVG(math_score),1) AS math,
       ROUND(AVG(science_score),1) AS science,
       ROUND(AVG(english_score),1) AS english
FROM students
"""))

# 8. Travel time, extra activities and gender vs score
print(q("""SELECT travel_time, ROUND(AVG(overall_score),1) AS avg_score FROM students GROUP BY travel_time"""))
print(q("""SELECT extra_activities, ROUND(AVG(overall_score),1) AS avg_score FROM students GROUP BY extra_activities"""))
print(q("""SELECT gender, ROUND(AVG(overall_score),1) AS avg_score FROM students GROUP BY gender"""))

# 9. Top 10 students by overall score who have attendance below 60%
print(q("""
SELECT student_id, attendance_percentage, overall_score, final_grade
FROM students
WHERE attendance_percentage < 60
ORDER BY overall_score DESC
LIMIT 10
"""))

  study_band  avg_score  students
0    0.5-1.9       39.8      2938
1      2-3.9       53.7      3885
2      4-5.9       69.5      4029
3        6-8       85.4      4148
  parent_education  avg_score  students
0          diploma       64.8      2581
1    post graduate       64.3      2535
2              phd       63.9      2426
3         graduate       63.9      2481
4        no formal       63.7      2445
5      high school       63.4      2532
   math  science  english
0  63.8     63.8     63.7
  travel_time  avg_score
0   15-30 min       64.1
1   30-60 min       63.6
2     <15 min       64.1
3     >60 min       64.3
  extra_activities  avg_score
0               no       64.0
1              yes       64.1
   gender  avg_score
0  female       64.3
1    male       64.0
2   other       63.8
   student_id  attendance_percentage  overall_score final_grade
0        4302                   56.8           95.9           a
1       14498                   58.8           95.4           a
2      

In [8]:
print(q("""
SELECT student_id, attendance_percentage, study_hours, overall_score
FROM students
WHERE attendance_percentage < 60
ORDER BY overall_score DESC
LIMIT 10
"""))

   student_id  attendance_percentage  study_hours  overall_score
0        4302                   56.8          7.9           95.9
1       14498                   58.8          8.0           95.4
2        9481                   59.5          7.8           94.8
3       12867                   57.2          8.0           94.5
4        5531                   58.6          7.7           94.3
5       11255                   57.6          7.9           94.3
6       13996                   59.3          7.7           94.2
7        2540                   58.1          7.9           94.0
8       14043                   55.0          7.8           94.0
9         591                   59.0          7.7           93.8
